# Tutorial Biopython

- **Conteúdo base:**
  - [Documentação do biopython](https://biopython.org/wiki/SeqIO)
  - [Tutorial e Cookbook](https://biopython.org/docs/latest/Tutorial/index.html)
- **Conteúdo adicional no GitHub:**
  - [peterjc/biopython_workshop](https://github.com/peterjc/biopython_workshop/blob/master/reading_sequence_files/README.rst)
  - [tiagoantao/biopython-notebook](https://github.com/tiagoantao/biopython-notebook)
  - [peterjc/biopython_workshop](https://github.com/peterjc/biopython_workshop)


## Conteúdo

- **Conceitos**:
  - Seq
  - SeqRecord
  - SeqIO
  - formatos: FASTA vs GenBank
- **Parse de GenBank**:
  - _SeqRecord.features_
  - extração de CDS/genes
  - metadados
- **Exercício prático**:
  1. Baixar um GenBank
  2. Extrair CDS
  3. Traduzir
  4. Salvar FASTA


## Preparo do material


Antes de começarmos, primeiro precisamos instalar o Biopython. E, em seguida, baixaremos os arquivos de sequência do NCBI para trabalharmos com eles durante o tutorial.


### Instalando Biopython


Para executarmos os códigos deste tutorial, precisamos instalar o Biopython. Você pode fazer isso usando pip, que é o gerenciador de pacotes do Python.

Ao utilizá-lo em um ambiente Jupyter Notebook, pode-se usar o comando mágico `%pip` para garantir que a instalação seja feita no ambiente correto. Além disso, podemos usar algumas opções (flags):

- `-q` ou `--quiet`: para reduzir a quantidade de saída durante a instalação, tornando o processo mais silencioso.
- `==1.86`: para especificar a versão do Biopython que queremos instalar


In [1]:
''' Instalando Biopython '''

%pip -q install biopython==1.86

Note: you may need to restart the kernel to use updated packages.


### Obtendo FASTA do NCBI


para trabalharmos com o Biopython, vamos precisar de um arquivo de sequência, um dos formatos mais comuns é o FASTA, que é um formato de texto simples para representar sequências biológicas. O Biopython tem uma funcionalidade integrada para baixar sequências do NCBI usando o módulo `Entrez`. A função `get_fasta_from_ncbi` abaixo exemplifica como fazer isso.


In [2]:
from Bio import Entrez  # Importando módulo para acessar o NCBI

In [3]:
''' Obter fasta '''


def save_fasta_from_entrez_to_file(accession: str, email: str = 'example@mail.com', db: str = 'nuccore') -> None:
    '''
        Baixa uma sequência FASTA do NCBI dado um número de acesso e salva em um arquivo
    '''
    Entrez.email = email
    received_accession = Entrez.efetch(
        db=db, id=accession, rettype="fasta", retmode="text")
    with received_accession as handle:

        with open(accession+'.fasta', 'w') as fasta_file:
            fasta_file.write(handle.read())


save_fasta_from_entrez_to_file('NC_045512.2', 'joaovitorfd2000@gmail.com')

## Leitura e Escrita de Sequências


- Interfaces de leitura e escrita de arquivos de sequência em [diversos formatos (FASTA, GenBank, etc.)](https://biopython.org/docs/latest/api/Bio.SeqIO.html#file-formats):
  - `Bio.SeqIO`: interação padronizada para diferentes formatos de sequência
    - Preferível pois é mais simples e trabalha de forma padrão com os objetos SeqRecord
  - `Bio.AlignIO`: interação específica para cada formato de alinhamento
    - É mais flexível, mas requer mais conhecimento sobre os formatos de alinhamento

> **Formatos:** abi, abi-trim, ace, cif-atom, cif-seqres, clustal, embl, fasta, fasta-2line, fastq-sanger or fastq, fastq-solexa, fastq-illumina, gck, genbank or gb, ig, imgt, nexus, pdb-seqres, pdb-atom, phd, phylip, pir, seqxml, sff, sff-trim, snapgene, stockholm, swiss, tab, qual, uniprot-xml, xdna.

---

- [ ] Sequência vs seqrecord


### SeqIO: interface padronizada


In [4]:
''' Importando módulos do Biopython para ilustrar leitura e escrita de sequências '''

from Bio import SeqIO  # importando módulos do Biopython
from Bio.SeqRecord import SeqRecord  # importando SeqRecord

### Bio.SeqIO.Parse: leitura de arquivos de sequência


Aqui indicamos o nome do arquivo e o formato, e o Biopython irá ler o arquivo e criar um objeto SeqRecord para cada sequência presente no arquivo. O método `parse` é um gerador, o que significa que ele retorna um iterador que pode ser usado para percorrer os registros um por um, sem carregar todo o arquivo na memória de uma vez. São apresentadas duas formas de usar o `parse`:

1. Diretamente, passando o nome do arquivo e o formato.
2. Abrindo o arquivo manualmente e passando o objeto de arquivo para o `parse`.

Em seguida, mostramos como usar o `read` para ler um único registro de um arquivo FASTA. O `read` é usado quando se espera que o arquivo contenha apenas um registro, e ele retorna esse registro como um objeto SeqRecord. Se o arquivo contiver mais de um registro, o `read` lançará uma exceção.


In [7]:
''' Processando o arquivo FASTA baixado '''

file_name = 'NC_045512.2.fasta'
file_format = 'fasta'

print('Records from...')

print('1. direct parsing')
for record in SeqIO.parse(file_name, file_format):
    print('\t- id:', record.id)

# Alternativamente:

print('2. file opening')
with open(file_name) as fasta_file:
    for record in SeqIO.parse(fasta_file, file_format):
        print('\t- id:', record.id)

print('3. first record parsing')
fasta_record = SeqIO.read(file_name, 'fasta')
print('\t- id:', fasta_record.id)

Records from...
1. direct parsing
	- id: NC_045512.2
2. file opening
	- id: NC_045512.2
3. first record parsing
	- id: NC_045512.2


Para registros pequenos, a otimização de memória do `parse` não é tão necessária, portanto, para facilitar o acesso aos dados, podemos convertê-lo em uma lista usando a função `list()`. Isso carregará todos os registros na memória, permitindo que você acesse cada um deles diretamente por meio de índices. No entanto, é importante ter cuidado ao usar essa abordagem com arquivos grandes, pois pode consumir muita memória. Outra alternativa é usar o `SeqIO.to_dict()`, que permite criar um dicionário de registros, onde as chaves são os IDs dos registros e os valores são os objetos SeqRecord correspondentes. Isso pode ser útil para acessar rapidamente um registro específico através de seu ID.


In [ ]:
'''  Lendo o arquivo inteiro para a memória '''

lista_de_records = list(SeqIO.parse(file_name, file_format))
print('Número de records lidos:', len(lista_de_records))
print('Último record id:', lista_de_records[-1].id)

record_dict = SeqIO.to_dict(SeqIO.parse(file_name, file_format))
print(record_dict)
# print('O record é: {\n' + str(record_dict['NC_045512.2']) + '\n}')

Número de records lidos: 1
Último record-id: NC_045512.2
{'NC_045512.2': SeqRecord(seq=Seq('ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTCTTGT...AAA'), id='NC_045512.2', name='NC_045512.2', description='NC_045512.2 Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome', dbxrefs=[])}


- SeqIO.to_dict vs SeqIO.index:
  - to_dict: carrega tudo na memória
  - index: carrega sob demanda

Para sequências muito grandes, sugere-se usar o BioSQL.
